# Data Analytics Lab Sheet 1: Real-World Data Collection, Cleaning and Preprocessing

**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  
**Program:** B.Tech CSE  

## Aim
Collect a public Titanic dataset and preprocess missing values, duplicates, categorical variables, outliers, and numerical features.

> Run cells from top to bottom. Dataset sources and any synthetic-data notes are documented in the relevant sections.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Load the Titanic dataset

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect the dataset

In [ ]:
display(df.head())
df.info()
display(df.describe(include="all").T)

## 4. Missing values and duplicates

In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))
print("Duplicate records:", df.duplicated().sum())

## 5. Handle missing values

In [ ]:
df["age"] = df["age"].fillna(df["age"].median())
for col in ["embarked", "embark_town"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])
if "deck" in df.columns:
    df = df.drop(columns=["deck"])
print("Missing values after treatment:")
display(df.isnull().sum())

## 6. Remove duplicate records

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)

## 7. Standardize categorical values and encode

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns
for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

label_encoder = LabelEncoder()
df["sex_encoded"] = label_encoder.fit_transform(df["sex"])
df = pd.get_dummies(df, columns=["embarked"], prefix="embarked", dtype=int)
display(df.head())

## 8. Detect and cap fare outliers

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["fare"])
plt.title("Fare Outliers")
plt.show()

q1, q3 = df["fare"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = (df["fare"] < lower_bound) | (df["fare"] > upper_bound)
print("Fare outliers:", int(outlier_mask.sum()))
df["fare"] = df["fare"].clip(lower=lower_bound, upper=upper_bound)

## 9. Normalize numerical features and engineer features

In [ ]:
scaler = MinMaxScaler()
df[["age", "fare"]] = scaler.fit_transform(df[["age", "fare"]])
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["age_group"] = pd.cut(
    df["age"], bins=[-0.01, 0.25, 0.60, 1.0],
    labels=["Young", "Adult", "Senior"]
)
display(df[["age", "fare", "family_size", "age_group"]].head())

## 10. Final inspection and export

In [ ]:
print("Final dataset shape:", df.shape)
display(df.isnull().sum())
display(df.head())
output_file = "Titanic_Cleaned_Preprocessed.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file)